In [1]:
male_keywords = ["подсудимый", "обвиняемый", "осуждённый", "обвиняемого", "подсудимого", "Подсудимый"]
female_keywords = ["подсудимая", "обвиняемая", "осуждённая", "подсудимой"]

def get_full_text(row):
    return f"{str(row['preamble'])} {str(row['description'])} {str(row['sentence'])}"

In [ ]:
import pandas as pd

df_full = pd.read_csv('df_with_marking_final_full.csv')

In [3]:
df_marking = df_full.sample(n=300, random_state=42).reset_index(drop=True)

In [4]:
train_data = []

for idx, row in df_marking.iterrows():
    text = get_full_text(row)
    text_lower = text.lower()
    gender = row["predicted_gender"]

    if gender == "неизвестно" or pd.isnull(gender):
        continue

    keywords = female_keywords if gender == "женщина" else male_keywords
    found = False

    for keyword in keywords:
        start = text_lower.find(keyword)
        if start != -1:
            end = start + len(keyword)
            train_data.append((text, {"entities": [(start, end, "GENDER_ACC")]}))
            found = True
            break

    if not found:
        print(f"Не удалось найти ключевое слово для пола '{gender}' в тексте id={row['id']}")

print(f"TRAIN_DATA готово: {len(train_data)} примеров")

Не удалось найти ключевое слово для пола 'женщина' в тексте id=104104
Не удалось найти ключевое слово для пола 'женщина' в тексте id=119588
Не удалось найти ключевое слово для пола 'женщина, женщина' в тексте id=21116
Не удалось найти ключевое слово для пола 'женщина' в тексте id=44339
TRAIN_DATA готово: 295 примеров


In [6]:
import spacy
from spacy.training.example import Example
from spacy.util import minibatch
import random

nlp = spacy.blank("ru")

ner = nlp.add_pipe("ner")

ner.add_label("GENDER_ACC")

examples = []
for text, annotations in train_data:
    doc = nlp.make_doc(text)
    examples.append(Example.from_dict(doc, annotations))

from spacy.util import compounding

n_iter = 15
optimizer = nlp.begin_training()

for i in range(n_iter):
    random.shuffle(examples)
    losses = {}
    batches = minibatch(examples, size=compounding(4.0, 32.0, 1.001))

    for batch in batches:
        nlp.update(batch, sgd=optimizer, losses=losses) 

    print(f"Итерация {i+1}/{n_iter} — потери: {losses}")

nlp.to_disk("ner_b_gender_model")
print("Модель сохранена в папке 'ner_b_gender_model'")

Итерация 1/15 — потери: {'ner': 136867.2657392293}
Итерация 2/15 — потери: {'ner': 262.9053459120047}
Итерация 3/15 — потери: {'ner': 189.5719321438859}
Итерация 4/15 — потери: {'ner': 142.17188988462416}
Итерация 5/15 — потери: {'ner': 103.26530545052837}
Итерация 6/15 — потери: {'ner': 81.47160447057165}
Итерация 7/15 — потери: {'ner': 60.437949724804184}
Итерация 8/15 — потери: {'ner': 50.52504885820011}
Итерация 9/15 — потери: {'ner': 34.60052255743908}
Итерация 10/15 — потери: {'ner': 43.39088000345748}
Итерация 11/15 — потери: {'ner': 29.649733101700026}
Итерация 12/15 — потери: {'ner': 34.136698778108126}
Итерация 13/15 — потери: {'ner': 26.92834847018167}
Итерация 14/15 — потери: {'ner': 25.09053011027627}
Итерация 15/15 — потери: {'ner': 31.569749309611748}
Модель сохранена в папке 'ner_b_gender_model'


In [30]:
df_test = pd.read_csv('df_with_marking_final.csv')

In [31]:
import spacy
from sklearn.metrics import accuracy_score, f1_score

nlp = spacy.load("ner_b_gender_model")

true_labels = []
pred_labels = []

def get_full_text(row):
    return f"{str(row['preamble'])} {str(row['description'])} {str(row['sentence'])}"

for idx, row in df_test.iterrows():
    true_gender = row["gender_accused"]
    if pd.isnull(true_gender) or true_gender == "неизвестно":
        continue  # Пропускаем "неизвестно"

    text = get_full_text(row)
    doc = nlp(text)

    predicted_gender = "неизвестно"
    for ent in doc.ents:
        if ent.label_ == "GENDER_ACC":
            word = ent.text.lower()
            # print(word)
            if word in female_keywords:
                predicted_gender = "женщина"
            elif word in male_keywords:
                predicted_gender = "мужчина"
            else:
                print(word)
            break

    true_labels.append(true_gender)
    pred_labels.append(predicted_gender)

accuracy = accuracy_score(true_labels, pred_labels)
f1 = f1_score(true_labels, pred_labels, average='weighted') 
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.3f}")


Accuracy: 0.7600
F1 Score: 0.847
